# Feature exploration: candidate features for `y_g12`

Fresh, exploratory pass over the **local gold** (`data/gold/facility_month_dataset/t=*`). Nothing here writes
to the repo or to `data/`; every result stays in this notebook.

**Question.** Do a few families of engineered features add signal on top of the gold model features?

**Rules kept** (project rules):
- Every candidate is computed from the **same row** (vintage t) only: no t-1/t-3 lags, no labels, no future rows.
  Peer ranks use the same-month cross-section within a state, which carries no label.
- Star ratings are never features, so `staffing_rating` and `qm_rating` are dropped even though
  `configs/feature_registry.yaml` marks them `model` (a conflict to raise with the team).
- Splits are the project's own g12 windows (`gold.splits.split_windows`, `configs/splits.yaml`). Feature
  decisions use **calib**; **test** is looked at once, at the end.
- Leakage screen: any feature that alone gets PR-AUC > 0.6 or > 3x the base rate is flagged.

Run with: `uv run --extra ml --group notebooks` (VS Code / Jupyter kernel from `.venv`).

In [ ]:
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import average_precision_score

from precursorintelligence.common.config import (
    load_feature_registry,
    load_labels_config,
    load_splits_config,
)
from precursorintelligence.gold.splits import split_windows

ROOT = Path.cwd()
while not (
    ROOT / "pyproject.toml"
).exists():  # works from notebooks/feature_exploration/ or the repo root
    ROOT = ROOT.parent
GOLD = ROOT / "data/gold/facility_month_dataset"
LABEL, V = "g12", "2026-08"  # V = v_labels in data/gold/_current.json
SEEDS = (0, 1, 2)
N_BOOT = 200
rng = np.random.default_rng(0)

## 1. Load gold and assign the g12 splits
The local gold predates the `split_*` columns, so the windows are recomputed from `configs/splits.yaml`
with the same function gold uses. Only `positive` / `negative` rows are used.

In [ ]:
raw = pl.read_parquet(str(GOLD / "*/*.parquet"), hive_partitioning=False)
labels_cfg, splits_cfg = load_labels_config(), load_splits_config()
w = split_windows(V, labels_cfg.labels[LABEL].horizon_months, labels_cfg.buffer(LABEL), splits_cfg)
print({k: w[k] for k in ("train", "calib", "purge", "test")})

t = pl.col("t")
split = (
    pl.when(t.is_between(*map(pl.lit, w["train"])))
    .then(pl.lit("train"))
    .when(t.is_between(*map(pl.lit, w["calib"])))
    .then(pl.lit("calib"))
    .when(t.is_between(*map(pl.lit, w["test"])))
    .then(pl.lit("test"))
    .otherwise(None)
)
df = raw.with_columns(split.alias("split")).filter(
    pl.col("split").is_not_null() & pl.col(f"status_{LABEL}").is_in(["positive", "negative"])
)
assert df["ccn"].str.len_chars().eq(6).all(), "CCN must be 6-character text"
print(
    df.group_by("split")
    .agg(
        pl.len().alias("rows"),
        pl.col("ccn").n_unique().alias("facilities"),
        pl.col(f"y_{LABEL}").mean().round(3).alias("base_rate"),
        pl.col("t").min().alias("t_min"),
        pl.col("t").max().alias("t_max"),
    )
    .sort("t_min")
)

## 2. Baseline feature set
Registry features with `model_use: model`, minus the star ratings.

In [ ]:
STAR_RATINGS = {"staffing_rating", "qm_rating", "health_inspection_rating", "overall_rating"}
registry = load_feature_registry()
BASE = [f.name for f in registry.features if f.model_use == "model" and f.name not in STAR_RATINGS]
print(len(BASE), "baseline features")

## 3. Candidate features (same-row only)

| Family | Idea |
|---|---|
| `momentum` | the last 12 months vs the 12 before, from the 12m and 24m windows in the same file |
| `mix` | what kind of citations: harm share, complaint share; size-normalised counts |
| `peer_rank` | percentile within the same state and month (staffing, turnover, citations, complaints) |
| `timing` | a standard survey due soon, alone and with a prior harm citation; a recent complaint survey |
| `staffing_stress` | weekend RN gap; low RN staffing together with high turnover |

In [ ]:
def ratio(num: pl.Expr, den: pl.Expr) -> pl.Expr:
    return pl.when(den > 0).then(num / den).otherwise(None)


def pct_rank(col: str) -> pl.Expr:
    return (pl.col(col).rank("average") / pl.col(col).count()).over("state", "t")


momentum = [
    (pl.col("n_cit_12m") - (pl.col("n_cit_24m") - pl.col("n_cit_12m"))).alias("cit_momentum_12m"),
    (pl.col("sev_points_12m") - (pl.col("sev_points_24m") - pl.col("sev_points_12m"))).alias(
        "sev_momentum_12m"
    ),
    (
        pl.col("n_complaint_surveys_cited_12m")
        - (pl.col("n_complaint_surveys_cited_24m") - pl.col("n_complaint_surveys_cited_12m"))
    ).alias("complaint_momentum_12m"),
]
mix = [
    ratio(pl.col("n_cit_GI_24m") + pl.col("n_cit_JL_24m"), pl.col("n_cit_24m")).alias(
        "harm_share_24m"
    ),
    ratio(pl.col("n_complaint_cit_12m"), pl.col("n_cit_12m")).alias("complaint_share_12m"),
    ratio(pl.col("n_cit_24m"), pl.col("certified_beds")).alias("cit_per_bed_24m"),
    ratio(100 * pl.col("n_complaint_surveys_cited_24m"), pl.col("certified_beds")).alias(
        "complaint_surveys_per_100beds_24m"
    ),
]
peer_rank = [
    pct_rank(c).alias(f"pr_{c}")
    for c in (
        "hprd_rn",
        "hprd_total",
        "turnover_total",
        "n_cit_24m",
        "n_complaint_surveys_cited_24m",
    )
]
due_soon = pl.col("days_to_std_due") <= 90  # includes overdue (negative)
prior_harm = pl.col("gplus_last_std") | (pl.col("n_gplus_surveys_36m") > 0)
timing = [
    due_soon.alias("std_due_90d"),
    (due_soon & prior_harm).alias("std_due_90d_x_prior_harm"),
    (pl.col("days_since_last_complaint_survey_cited") <= 90)
    .fill_null(False)
    .alias("complaint_survey_90d"),
]
staffing_stress = [
    (pl.col("hprd_rn") - pl.col("hprd_weekend_rn")).alias("weekend_rn_gap"),
    ((pct_rank("hprd_rn") < 0.25) & (pct_rank("turnover_total") > 0.75)).alias(
        "low_rn_high_turnover"
    ),
]
FAMILIES = {
    "momentum": momentum,
    "mix": mix,
    "peer_rank": peer_rank,
    "timing": timing,
    "staffing_stress": staffing_stress,
}
# Peer ranks rank within (state, t) over every facility in that month, labelled or not, as at scoring time.
eng = raw.with_columns([e for exprs in FAMILIES.values() for e in exprs])
NEW = {fam: [e.meta.output_name() for e in exprs] for fam, exprs in FAMILIES.items()}
df = df.join(
    eng.select("ccn", "t", *[c for cols in NEW.values() for c in cols]), on=["ccn", "t"], how="left"
)
print({fam: cols for fam, cols in NEW.items()})

## 4. Leakage screen and single-feature signal (train only)
Each candidate alone, as a ranking score (sign chosen on train). Flag: PR-AUC > 0.6 or > 3x the base rate.

In [ ]:
train = df.filter(pl.col("split") == "train")
y_tr = train[f"y_{LABEL}"].to_numpy()
base_rate = y_tr.mean()
rows = []
for fam, cols in NEW.items():
    for c in cols:
        x = train[c].cast(pl.Float64).to_numpy()
        ok = ~np.isnan(x)
        ap = max(
            average_precision_score(y_tr[ok], x[ok]), average_precision_score(y_tr[ok], -x[ok])
        )
        rows.append(
            {
                "family": fam,
                "feature": c,
                "non_null": ok.mean().round(3),
                "pr_auc": round(ap, 3),
                "lift": round(ap / y_tr[ok].mean(), 2),
                "flag": ap > 0.6 or ap > 3 * y_tr[ok].mean(),
            }
        )
signal = pd.DataFrame(rows).sort_values("pr_auc", ascending=False)
print(f"train base rate {base_rate:.3f}")
signal

## 5. Ablation on calib: baseline + one family at a time
LightGBM with fixed parameters (no tuning), predictions averaged over 3 seeds. ΔPR-AUC vs the baseline with a
95% interval from a **facility-level** paired bootstrap on calib (rows of one facility move together).

**Calib levels are optimistic.** `configs/splits.yaml` puts no purge gap between train and calib, so their
12-month label windows overlap and the same facilities appear in both. Read the *deltas* here; the purged
test month in section 7 gives the honest level.

In [ ]:
CATEGORICAL = [f.name for f in registry.features if f.dtype == "str" and f.name in BASE]
PARAMS = dict(
    objective="binary",
    learning_rate=0.05,
    n_estimators=400,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    verbose=-1,
    n_jobs=-1,
)


def to_pandas(frame: pl.DataFrame, cols: list[str]) -> pd.DataFrame:
    X = frame.select(cols).to_pandas()
    for c in cols:
        if c in CATEGORICAL:
            X[c] = X[c].astype("category")
        elif X[c].dtype == bool or X[c].dtype == object:
            X[c] = X[c].astype("float")
    return X


def fit_predict(cols: list[str], fit_on: pl.DataFrame, score_on: pl.DataFrame):
    Xf, Xs = to_pandas(fit_on, cols), to_pandas(score_on, cols)
    for c in cols:  # categories seen in training only
        if c in CATEGORICAL:
            Xs[c] = pd.Categorical(Xs[c], categories=Xf[c].cat.categories)
    preds, models = [], []
    for s in SEEDS:
        m = lgb.LGBMClassifier(random_state=s, **PARAMS).fit(Xf, fit_on[f"y_{LABEL}"].to_numpy())
        preds.append(m.predict_proba(Xs)[:, 1])
        models.append(m)
    return np.mean(preds, axis=0), models


def precision_at(y, p, frac):
    k = max(1, int(len(p) * frac))
    return y[np.argsort(-p)[:k]].mean()


calib = df.filter(pl.col("split") == "calib")
y_cal = calib[f"y_{LABEL}"].to_numpy()
fac_idx = calib["ccn"].to_pandas().groupby(calib["ccn"].to_pandas()).indices
fac_rows = list(fac_idx.values())
BOOT = [
    np.concatenate([fac_rows[i] for i in rng.integers(0, len(fac_rows), len(fac_rows))])
    for _ in range(N_BOOT)
]


def delta_ci(p_new, p_base, y):
    d = [
        average_precision_score(y[b], p_new[b]) - average_precision_score(y[b], p_base[b])
        for b in BOOT
    ]
    return np.percentile(d, [2.5, 97.5])


p_base, _ = fit_predict(BASE, train, calib)
ap_base = average_precision_score(y_cal, p_base)
results = [
    {
        "set": "baseline",
        "n_features": len(BASE),
        "pr_auc": ap_base,
        "p_at_10": precision_at(y_cal, p_base, 0.10),
        "delta": 0.0,
        "ci_low": np.nan,
        "ci_high": np.nan,
    }
]
preds = {"baseline": p_base}
for fam, cols in {**NEW, "all": [c for cols in NEW.values() for c in cols]}.items():
    p, _ = fit_predict(BASE + cols, train, calib)
    preds[fam] = p
    lo, hi = delta_ci(p, p_base, y_cal)
    ap = average_precision_score(y_cal, p)
    results.append(
        {
            "set": f"+ {fam}",
            "n_features": len(BASE) + len(cols),
            "pr_auc": ap,
            "p_at_10": precision_at(y_cal, p, 0.10),
            "delta": ap - ap_base,
            "ci_low": lo,
            "ci_high": hi,
        }
    )
ablation = pd.DataFrame(results).round(4)
print(f"calib base rate {y_cal.mean():.3f}; a family helps only if ci_low > 0")
ablation

## 6. Where the new features land in the full model
Gain importance (mean of 3 seeds) for baseline + all families, new features marked.

In [ ]:
ALL = BASE + [c for cols in NEW.values() for c in cols]
_, models = fit_predict(ALL, train, calib)
gain = pd.Series(
    np.mean([m.booster_.feature_importance("gain") for m in models], axis=0), index=ALL
)
imp = (gain / gain.sum()).sort_values(ascending=False).rename("gain_share").to_frame()
imp["rank"] = range(1, len(imp) + 1)
imp["new"] = imp.index.isin(ALL[len(BASE) :])
imp[imp["new"]].round(4)

## 7. Test, looked at once
Fit on train + calib, score test. Baseline vs the families whose calib ci_low > 0 (or `all` if none qualifies,
reported for information only). This is a read-out, not a selection step.

In [ ]:
keep = [r["set"][2:] for r in results[1:] if r["ci_low"] > 0 and r["set"] != "+ all"]
chosen = [c for fam in keep for c in NEW[fam]] if keep else ALL[len(BASE) :]
fit_all = pl.concat([train, calib])
test = df.filter(pl.col("split") == "test")
y_te = test[f"y_{LABEL}"].to_numpy()
p0, _ = fit_predict(BASE, fit_all, test)
p1, _ = fit_predict(BASE + chosen, fit_all, test)
print(
    f"test rows {len(y_te)}, base rate {y_te.mean():.3f}; families kept on calib: {keep or 'none (showing all)'}"
)
readout = pd.DataFrame(
    {
        "pr_auc": [average_precision_score(y_te, p0), average_precision_score(y_te, p1)],
        "p_at_5": [precision_at(y_te, p0, 0.05), precision_at(y_te, p1, 0.05)],
        "p_at_10": [precision_at(y_te, p0, 0.10), precision_at(y_te, p1, 0.10)],
    },
    index=["baseline", f"baseline + {len(chosen)} new"],
).round(4)
readout

## 8. Notes and next steps
- **Lags are not possible from gold rows.** The spine starts in 2023-01, so a t-6 or t-12 gold row does not
  exist for most training months. Deltas belong in the gold build, from silver vintages t-6 / t-12 (as
  `d6_hprd_total` already does); `d6`/`d12` of citations or turnover would be the next candidates there.
- Peer ranks are computed over every facility in the state-month, so they also work at scoring time.
- A family that clears calib should go into `src/precursorintelligence/features/` with a registry entry and a
  leakage test before any model uses it.